# 第6回 画像認識 — 畳み込みニューラルネットワーク

**このノートブックは第6回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 画像が「数字の並び」であることを確かめる
2. フィルタを手書き数字に当てて、何が光るか見る
3. MLP と CNN を学習させ、重みの数と正解率を比べる
4. 「CNN は平行移動に強い」がどこまで本当かを測る

所要時間:実習 約25分 / 課題 約40分

> **注意**:学習に2〜4分かかるセルがあります。実行中は左の丸印が回ります。

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。
最初の2つのセルは、合わせて1〜2分かかります。

In [ ]:
# 準備(このセルを最初に1回だけ実行。1〜2分かかります)
!pip install -q japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, japanize_matplotlib
from torchvision import datasets, transforms
plt.rcParams["font.size"] = 12
torch.manual_seed(0)
print("準備できました / PyTorch", torch.__version__)

In [ ]:
# 手書き数字(MNIST)を読み込む。初回は数十秒かかります。
訓練元 = datasets.MNIST("./data", train=True,  download=True, transform=transforms.ToTensor())
試験元 = datasets.MNIST("./data", train=False, download=True, transform=transforms.ToTensor())

def まとめる(データ, 枚数):
    X = torch.stack([データ[i][0] for i in range(枚数)])
    y = torch.tensor([データ[i][1] for i in range(枚数)])
    return X, y

# 授業中に回せるよう、6000枚だけ使います(全部だと6万枚)
訓練X, 訓練y = まとめる(訓練元, 6000)
試験X, 試験y = まとめる(試験元, 2000)
print("訓練", tuple(訓練X.shape), " テスト", tuple(試験X.shape))

fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(訓練X[i][0], cmap="gray_r"); ax.set_title(int(訓練y[i])); ax.axis("off")
plt.suptitle("こういう画像を見分けます(28×28 の白黒)")
plt.tight_layout(); plt.show()

## 1. 画像は、ただの数字の並び

AI は「7 という形」を見ていません。28×28 = **784個の数字**を受け取っているだけです。
形も、線も、上下左右も、最初は何も分かっていません。

In [ ]:
# ===== ① 画像は、ただの数字の並び =====
画像 = 訓練X[0][0]
print("形:", tuple(画像.shape), " = ", 画像.numel(), "個の数")
print("いちばん暗い値", float(画像.min()), " いちばん明るい値", float(画像.max()))
print()
print("中央の 10×10 だけを数字で表示します(0.00 が白、1.00 が黒):")
for 行 in 画像[9:19, 9:19]:
    print("  " + " ".join(f"{float(v):.1f}" for v in 行))
print()
print("AI が受け取るのは、この数字だけです。「7 という形」ではありません。")

## 2. フィルタを当ててみる

**畳み込み**とは、3×3 の小さな窓(フィルタ)を画像の上で滑らせて、
重なった9個の数を **掛けて足す** 操作です。それだけです。

まず、人間が手で作ったフィルタを当てて、何が起きるか見てみましょう。

In [ ]:
# ===== ② フィルタを当ててみる =====
フィルタ = {
    "縦線を探す":  [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],
    "横線を探す":  [[-1, -1, -1], [0, 0, 0], [1, 1, 1]],
    "ぼかす":      [[1/9]*3, [1/9]*3, [1/9]*3],
}

def 当てる(画像, カーネル):
    x = 画像.reshape(1, 1, 28, 28)
    k = torch.tensor(カーネル, dtype=torch.float32).reshape(1, 1, 3, 3)
    return F.conv2d(x, k).reshape(26, 26)

見本 = 訓練X[0][0]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(見本, cmap="gray_r"); axes[0].set_title("もとの画像"); axes[0].axis("off")
for ax, (名前, k) in zip(axes[1:], フィルタ.items()):
    結果 = 当てる(見本, k)
    ax.imshow(結果, cmap="gray_r"); ax.set_title(名前); ax.axis("off")
plt.tight_layout(); plt.show()

print("縦線フィルタは縦向きのふちが、横線フィルタは横向きのふちが光ります。")
print("ぼかしフィルタは何も強調しません(全部を平均するだけだから)。")

> フィルタごとに、**同じ画像から違うものが取り出せる**ことに注目してください。
> だからフィルタは1枚では足りません。今日の CNN は1層目に8枚使います。

In [ ]:
# 授業の例題と同じ計算を、手で確かめる
画像 = torch.tensor([[0., 0., 0., 0., 0.],
                     [0., 0., 0., 0., 0.],
                     [9., 9., 9., 9., 9.],
                     [0., 0., 0., 0., 0.],
                     [0., 0., 0., 0., 0.]])
横線 = torch.tensor([[-1., -1., -1.], [0., 0., 0.], [1., 1., 1.]])

窓 = 画像[0:3, 0:3]
print("左上の 3×3 の窓:"); print(窓.numpy())
print("フィルタ:");        print(横線.numpy())
print()
print("掛けたもの:");      print((窓 * 横線).numpy())
print("全部足すと:", float((窓 * 横線).sum()), " ← 授業で計算した 27 と一致")
print()
出力 = F.conv2d(画像.reshape(1,1,5,5), 横線.reshape(1,1,3,3)).reshape(3,3)
print("全部の位置に当てた結果(3×3):"); print(出力.numpy())
print()
print("上の段が +27、下の段が −27。同じ横線でも、窓のどこに来るかで符号が変わります。")
print("符号 = 境目の向き、大きさ = 境目の強さ です。")

## 3. MLP と CNN を比べる

- **MLP**(全結合)… 784個の画素を全部つなぐ、素直なやり方
- **CNN**(畳み込み)… 小さなフィルタを画像じゅうで使い回すやり方

重みの数と正解率を比べます。

In [ ]:
# ===== ③ MLP と CNN を比べる =====
def 作るMLP():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, 128), nn.ReLU(),
        nn.Linear(128, 10))

def 作るCNN():
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 28 → 14
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 14 → 7
        nn.Flatten(),
        nn.Linear(16*7*7, 10))

def 重みの数(m):
    return sum(p.numel() for p in m.parameters())

print(f"MLP の重み {重みの数(作るMLP()):>8,} 個")
print(f"CNN の重み {重みの数(作るCNN()):>8,} 個")
print(f"→ CNN のほうが {重みの数(作るMLP())/重みの数(作るCNN()):.1f} 分の1\n")

print("CNN の内訳(どの層が重みを使っているか):")
for 名前, p in 作るCNN().named_parameters():
    if "weight" in 名前:
        print(f"  {名前:<12}{str(tuple(p.shape)):<18}{p.numel():>8,} 個")
print("\n畳み込みではなく、最後の全結合がいちばん重いことに注目してください。")

In [ ]:
# 学習させる(2つ合わせて2〜4分かかります)
def 学習(作る関数, 周回=12, 拡張=False, seed=0):
    torch.manual_seed(seed)
    モデル = 作る関数()
    最適化 = torch.optim.Adam(モデル.parameters(), lr=1e-3)
    損失関数 = nn.CrossEntropyLoss()
    for 周 in range(周回):
        順番 = torch.randperm(len(訓練X))
        for i in range(0, len(訓練X), 64):
            番号 = 順番[i:i+64]
            xb = 訓練X[番号]
            if 拡張:                        # ランダムにずらして見せる
                dx, dy = torch.randint(-3, 4, (2,))
                xb = torch.roll(xb, shifts=(int(dy), int(dx)), dims=(2, 3))
            最適化.zero_grad()
            損失関数(モデル(xb), 訓練y[番号]).backward()
            最適化.step()
    return モデル

def 正解率(モデル, X, y):
    モデル.eval()
    with torch.no_grad():
        return float((モデル(X).argmax(1) == y).float().mean()) * 100

MLP = 学習(作るMLP)
CNN = 学習(作るCNN)
print(f"MLP: 重み {重みの数(MLP):>8,} 個   正解率 {正解率(MLP, 試験X, 試験y):.1f}%")
print(f"CNN: 重み {重みの数(CNN):>8,} 個   正解率 {正解率(CNN, 試験X, 試験y):.1f}%")
print("\n重みが11分の1なのに、CNN のほうが成績が上のはずです。")

## 4. 教科書を疑う — 「CNN は平行移動に強い」は本当か

教科書には「プーリングがあるから CNN は位置のずれに強い」と書いてあります。
**本当でしょうか。測ってみましょう。**

テスト画像を右にずらして、正解率がどう変わるかを見ます。
比較のため、訓練時にランダムにずらして見せた CNN(**データ拡張**)も作ります。

In [ ]:
# ===== ④ 教科書を疑う:ずらすとどうなるか =====
# 訓練時にランダムにずらして見せたモデルも作ります(データ拡張)
CNN拡張 = 学習(作るCNN, 拡張=True)

def ずらす(X, 画素):
    return torch.roll(X, shifts=(0, 画素), dims=(2, 3))

ずらし量 = list(range(6))
記録 = {"全結合(MLP)": [], "畳み込み(CNN)": [], "CNN + データ拡張": []}
for d in ずらし量:
    Xs = ずらす(試験X, d)
    記録["全結合(MLP)"].append(正解率(MLP, Xs, 試験y))
    記録["畳み込み(CNN)"].append(正解率(CNN, Xs, 試験y))
    記録["CNN + データ拡張"].append(正解率(CNN拡張, Xs, 試験y))

print(f"{'ずらし':>7}{'MLP':>10}{'CNN':>10}{'CNN+拡張':>12}")
for i, d in enumerate(ずらし量):
    print(f"{d:>5}px{記録['全結合(MLP)'][i]:>9.1f}%{記録['畳み込み(CNN)'][i]:>9.1f}%"
          f"{記録['CNN + データ拡張'][i]:>11.1f}%")

In [ ]:
# グラフで見る
plt.figure(figsize=(8.5, 4.6))
色 = {"全結合(MLP)": "#A8AECC", "畳み込み(CNN)": "#3A46A8", "CNN + データ拡張": "#E08A2B"}
for 名前, 値 in 記録.items():
    plt.plot(ずらし量, 値, "o-", lw=2.2, label=名前, color=色[名前])
plt.axhline(10, ls="--", c="gray", lw=1)
plt.text(0.1, 12, "でたらめに答えたときの正解率(10%)", color="gray", fontsize=10)
plt.xlabel("画像を右にずらした量(画素)"); plt.ylabel("正解率(%)")
plt.title("「CNN は平行移動に強い」は、どこまで本当か")
plt.legend(); plt.grid(alpha=.3); plt.ylim(0, 100)
plt.tight_layout(); plt.show()

print("読み取ってほしいこと:")
print("・MLP は急降下する。CNN はゆるやかだが、確実に下がる。")
print("・つまり「CNN は強い」は MLP と比べての話であって、「ずれても平気」ではない。")
print("・いちばん効いたのは、訓練時にずらして見せておくこと(データ拡張)。")

In [ ]:
# ずらした画像が、どう見えているのか
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for ax, d in zip(axes, ずらし量):
    ax.imshow(ずらす(試験X, d)[0][0], cmap="gray_r")
    ax.set_title(f"{d}px"); ax.axis("off")
plt.suptitle("人間にはどれも同じ数字に見えます")
plt.tight_layout(); plt.show()

for d in [0, 4]:
    予測 = int(CNN(ずらす(試験X, d)[0:1]).argmax())
    print(f"{d}px ずらした画像 → CNN の答え「{予測}」(正解は {int(試験y[0])})")

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) 自分でフィルタを作って当ててみる
私のフィルタ = [[0, -1, 0],
                [-1, 4, -1],
                [0, -1, 0]]          # ← ここを書きかえる(輪郭を出すフィルタ)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
axes[0].imshow(訓練X[3][0], cmap="gray_r"); axes[0].set_title("もと"); axes[0].axis("off")
axes[1].imshow(当てる(訓練X[3][0], 私のフィルタ), cmap="gray_r")
axes[1].set_title("自分のフィルタ"); axes[1].axis("off")
plt.tight_layout(); plt.show()

# (b) 上下にずらすとどうなるか(授業では左右だけ試しました)
def 上下にずらす(X, 画素):
    return torch.roll(X, shifts=(画素, 0), dims=(2, 3))

print(f"{'上下ずらし':>10}{'CNN':>10}{'CNN+拡張':>12}")
for d in [0, 2, 4]:
    Xs = 上下にずらす(試験X, d)
    print(f"{d:>8}px{正解率(CNN, Xs, 試験y):>9.1f}%{正解率(CNN拡張, Xs, 試験y):>11.1f}%")

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:フィルタを手で当てる
次の 4×4 の画像に、3×3 の「横線フィルタ」を当てて、2×2 の答えを手で計算しなさい。

```
画像                 フィルタ
9  9  9  9          −1  −1  −1
0  0  0  0           0   0   0
0  0  0  0          +1  +1  +1
0  0  0  0
```

- (a) 4か所ぶんを計算しなさい(上の窓が2か所、下の窓が2か所)
- (b) 答えの符号(プラス/マイナス)は何を意味しますか。**0 になる場所は何を意味しますか**

### 課題2:ノートブックを動かして報告する
- (a) 3つのフィルタを当てた画像を貼り、それぞれ何に反応しているか説明する
- (b) ずらし量 0〜5px に対する正解率のグラフを貼る(MLP / CNN / CNN+拡張 の3本)

### 課題3:次の主張を批判する(300字以上)

> 「工場の検品を CNN で自動化しました。テストデータで 99.2% の正解率が出ています。
> CNN は位置のずれに強いので、カメラの取り付け位置が多少変わっても問題ありません。」

考える材料:
- カメラが数ミリ動くと、画像では**何画素**ずれますか。今日測ったのは何画素までですか
- その 99.2% は、**どんな画像**で測った数字ですか(第2回・第5回)
- 対策はありますか。その対策に**代償**はありますか
- 検品では、見逃しと空振りのどちらが痛いですか。1つの数字で判断できますか(第3回)

**加点**:AI 以外の解決策(カメラを固定する、治具で位置を揃える等)に触れられると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。
うまくいかなかった結果も、なぜそうなったかを書いてあれば評価します。